#### run this command as needed to re-establish connection to Big Query
gcloud auth login --update-adc

### Set up and getting data

In [1]:
# !pip install xgboost
# !pip install imbalanced-learn
# !pip install catboost
# !pip install optuna
# !pip install torch
import pandas as pd
import numpy as np
from google.cloud import bigquery
import matplotlib.pyplot as plt
from sklearn.feature_selection import RFECV
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'

client = bigquery.Client()

In [7]:
# Get features
sql = """
SELECT
    a.*
FROM
    `anbc-hcb-prod.cm_medicaid_hcb_prod.medicaid_core2_0_final_score_history` AS a
LEFT JOIN
    `anbc-hcb-prod.cm_medicaid_hcb_prod.medicaid_strat_core_2_0_stratification_history` AS b
        ON a.asdb_member_key = b.asdb_member_key
LEFT JOIN
    `` AS c
        ON a.asdb_member_key = c.asdb_member_key
/*
LEFT JOIN
    `anbc-hcb-dev.cm_medicaid_hcb_dev.a534354_IP_2024_OOT_non_embedding_features_v2` AS c
        ON e.asdb_member_key = f.asdb_member_key
LEFT JOIN
    `anbc-hcb-dev.cm_medicaid_hcb_dev.a534354_IP_2024_OOT_other_cost_utilization_yr1` AS d
        ON e.asdb_member_key = g.asdb_member_key
LEFT JOIN
    `anbc-hcb-dev.cm_medicaid_hcb_dev.a534354_IP_2024_OOT_outcome_ip` AS e
        ON e.asdb_member_key = h.asdb_member_key
*/
WHERE 1=1
    AND NOT e.asdb_plan_key IN (33, 54)
    AND post_mnths >= 6
    AND e.asdb_elig_dt = "2023-11-01"
"""
df = client.query(sql).to_dataframe() 

df.shape
#2,191,619 members who qualify per BQ
#1,875,523 have embeddings
#1,647,108 after we cut out too short of followup time

(1641523, 54)

In [8]:
df.head()

,asdb_member_key,asdb_plan_key,plan_code,asdb_elig_dt,coa_population_category,agenbr,fostercare_ind,coa_population_group,ss_cohort,baseline,...,sum_ip_pre,ip_flag_pre,pcp_pre,spec_pre,paid_pre,ip_cost_pre,ed_cost_pre,op_flag_pre,ip_flag_post,sum_ip_post
0,460995305,46,FL,2023-11-01,CHIP,13,0,TANF/CHIP,Medicaid Kid,0.4,...,0.0,0,1,1,0E-9,0E-9,0E-9,0E-9,0,0.0
1,590157530,59,KS,2023-11-01,TANF,8,0,TANF/CHIP,Medicaid Kid,54.2,...,0.0,0,0,0,664.000000000,0E-9,0E-9,664.000000000,0,0.0
2,510361613,51,MI,2023-11-01,TANF,7,0,TANF/CHIP,Medicaid Kid,0.1,...,0.0,0,<NA>,<NA>,297.000000000,0E-9,0E-9,297.000000000,0,0.0
3,450841663,45,KY,2023-11-01,Expansion,54,0,Expansion,Medicaid Adult,90.1,...,0.0,0,1,6,4241.000000000,0E-9,1048.000000000,3193.000000000,0,0.0
4,590041224,59,KS,2023-11-01,TANF,17,0,TANF/CHIP,Medicaid Kid,45.9,...,0.0,0,0,0,807.000000000,0E-9,0E-9,807.000000000,0,0.0


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1641523 entries, 0 to 1641522
Data columns (total 54 columns):
 #   Column                   Non-Null Count    Dtype  
---  ------                   --------------    -----  
 0   asdb_member_key          1641523 non-null  Int64  
 1   asdb_plan_key            1641523 non-null  Int64  
 2   plan_code                1641523 non-null  object 
 3   asdb_elig_dt             1641523 non-null  object 
 4   coa_population_category  1641523 non-null  object 
 5   agenbr                   1641523 non-null  Int64  
 6   fostercare_ind           1641523 non-null  Int64  
 7   coa_population_group     1641523 non-null  object 
 8   ss_cohort                1641523 non-null  object 
 9   baseline                 1641523 non-null  float64
 10  ip_rnk                   1641523 non-null  float64
 11  cdpsrx_rnk               1641523 non-null  float64
 12  ip_risk                  1641523 non-null  float64
 13  cdpsrx_scr               1641523 non-null 